# 🔬 Mitigating Hallucination and Perception Errors in Vision Language Agents

**Author:** Vedhagiri Alagesan | **Supervisor:** Dr. Oliver Lemon | **University:** Heriot-Watt University

This notebook implements the complete pipeline:
1. **Module 1:** Baseline VLM + YOLOv8 Detection + Data Loading
2. **Module 2:** Mitigation Strategies (Grounded Prompting, CoT, Self-Verification)
3. **Module 3:** GRPO Training with Hallucination-Aware Rewards
4. **Module 4:** Ablation Studies & Evaluation
5. **Module 5:** Results Visualization

**Runtime:** Google Colab with A100 GPU

---
## 0. Environment Setup
Clone the repository and install all dependencies.

In [ ]:
# ============================================================
# 0.1 Clone Repository (update with your GitHub URL)
# ============================================================
import os

REPO_URL = "https://github.com/Gsam0612/Mitigating-Hallucination-and-Perception-Error.git/"
PROJECT_DIR = "/content/Mitigating-Hallucination-and-Perception-Error"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 0.2 Install Dependencies
# ============================================================
!pip install -q torch torchvision transformers accelerate peft trl bitsandbytes
!pip install -q ultralytics pycocotools datasets
!pip install -q scikit-learn nltk matplotlib seaborn pandas pyyaml tqdm wandb jsonlines

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# 0.3 Download Datasets
# ============================================================
import os

# --- COCO 2017 Validation Set ---
COCO_DIR = "./data/coco"
os.makedirs(f"{COCO_DIR}/annotations", exist_ok=True)

if not os.path.exists(f"{COCO_DIR}/val2017"):
    print("Downloading COCO 2017 validation images...")
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
    !unzip -q /tmp/val2017.zip -d {COCO_DIR}
    !rm /tmp/val2017.zip
    print("COCO images downloaded.")

if not os.path.exists(f"{COCO_DIR}/annotations/instances_val2017.json"):
    print("Downloading COCO annotations...")
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip
    !unzip -q -o /tmp/ann.zip -d {COCO_DIR}
    !rm /tmp/ann.zip
    print("COCO annotations downloaded.")

# --- POPE Benchmark ---
POPE_DIR = "./data/pope"
os.makedirs(POPE_DIR, exist_ok=True)

if not os.path.exists(f"{POPE_DIR}/coco_pope_random.json"):
    print("Downloading POPE benchmark...")
    POPE_BASE = "https://raw.githubusercontent.com/AoiDragon/POPE/main/output/coco"
    for cat in ["random", "popular", "adversarial"]:
        !wget -q {POPE_BASE}/coco_pope_{cat}.json -O {POPE_DIR}/coco_pope_{cat}.json
    print("POPE benchmark downloaded.")

print("\n--- Dataset Summary ---")
print(f"COCO images: {len(os.listdir(f'{COCO_DIR}/val2017'))} files")
print(f"POPE files: {os.listdir(POPE_DIR)}")

In [ ]:
# ============================================================
# 0.4 Load Configuration
# ============================================================
import yaml
import sys

# Add project to path
sys.path.insert(0, PROJECT_DIR)

with open("config/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Update paths for Colab
cfg["data"]["coco_dir"] = COCO_DIR
cfg["data"]["coco_ann_file"] = f"{COCO_DIR}/annotations/instances_val2017.json"
cfg["data"]["coco_images_dir"] = f"{COCO_DIR}/val2017"
cfg["data"]["pope_dir"] = POPE_DIR

# Set seed for reproducibility
import random
import numpy as np
import torch

SEED = cfg["project"]["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration loaded. Seed set to:", SEED)

---
## Module 1: Foundation & Baseline
Load data, initialize models, and measure baseline hallucination rates.

In [ ]:
# ============================================================
# 1.1 Load COCO Ground Truth & Create Datasets
# ============================================================
from src.data.coco_loader import COCOGroundTruth, create_train_eval_split

coco_gt = COCOGroundTruth(
    ann_file=cfg["data"]["coco_ann_file"],
    images_dir=cfg["data"]["coco_images_dir"],
)

train_dataset, eval_dataset = create_train_eval_split(
    coco_gt,
    train_ratio=cfg["data"]["train_split"],
    max_train=cfg["data"]["max_train_samples"],
    max_eval=cfg["data"]["max_eval_samples"],
    seed=SEED,
)

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

# Quick sanity check
sample = train_dataset[0]
print(f"\nSample image ID: {sample['image_id']}")
print(f"Question: {sample['question']}")
print(f"GT objects: {sample['ground_truth']['unique_objects']}")
print(f"GT counts: {sample['ground_truth']['object_counts']}")

In [ ]:
# ============================================================
# 1.2 Initialize Baseline VLM (LLaVA-1.5-7B)
# ============================================================
from src.models.vlm_baseline import BaselineVLM

vlm = BaselineVLM(
    model_name=cfg["vlm"]["model_name"],
    load_in_4bit=cfg["vlm"]["load_in_4bit"],
    max_new_tokens=cfg["vlm"]["max_new_tokens"],
    temperature=cfg["vlm"]["temperature"],
)

In [ ]:
# ============================================================
# 1.3 Initialize YOLOv8 Object Detector
# ============================================================
from src.models.yolo_detector import YOLODetector

detector = YOLODetector(
    model_name=cfg["detector"]["model_name"],
    confidence_threshold=cfg["detector"]["confidence_threshold"],
    iou_threshold=cfg["detector"]["iou_threshold"],
    max_detections=cfg["detector"]["max_detections"],
)

In [ ]:
# ============================================================
# 1.4 Baseline Evaluation — Measure Hallucination BEFORE Mitigation
# ============================================================
from src.evaluation.hallucination_metrics import HallucinationMetrics, aggregate_metrics
from tqdm.notebook import tqdm

metrics_calculator = HallucinationMetrics()

print("Running baseline evaluation (no mitigation)...")
baseline_results = []
NUM_BASELINE_EVAL = 50  # Keep small for speed; increase for final run

for idx in tqdm(range(min(NUM_BASELINE_EVAL, len(eval_dataset)))):
    sample = eval_dataset[idx]
    image = sample["image"]
    question = sample["question"]
    gt = sample["ground_truth"]

    # Generate baseline response (no grounding, no CoT)
    response = vlm.generate(image, question)

    # Compute metrics
    m = metrics_calculator.compute_all_metrics(
        response=response,
        gt_objects=gt["unique_objects"],
        gt_counts=gt.get("object_counts"),
        gt_spatial=sample.get("spatial_relations"),
    )
    m["response"] = response
    m["image_id"] = sample["image_id"]
    baseline_results.append(m)

baseline_agg = aggregate_metrics(baseline_results)

print("\n" + "="*60)
print("BASELINE RESULTS (No Mitigation)")
print("="*60)
for k, v in baseline_agg.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 1.5 Visualize Some Baseline Examples
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image

os.makedirs("./results", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i, ax in enumerate(axes.flat):
    if i >= len(baseline_results):
        break
    sample = eval_dataset[i]
    result = baseline_results[i]

    ax.imshow(sample["image"])
    halluc = result.get("hallucinated_list", [])
    missed = result.get("missed_list", [])

    title = f"ID: {sample['image_id']}\n"
    title += f"Halluc: {halluc[:3]}\n" if halluc else "No hallucinations\n"
    title += f"Missed: {missed[:3]}" if missed else "None missed"

    color = "red" if halluc else "green"
    ax.set_title(title, fontsize=9, color=color)
    ax.axis("off")

plt.suptitle("Baseline VLM — Hallucination Examples", fontsize=14)
plt.tight_layout()
plt.savefig("./results/baseline_examples.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Module 2: Enhancement Strategies
Apply grounded prompting, chain-of-thought, and self-verification.

In [ ]:
# ============================================================
# 2.1 Initialize Grounded VLM (Detection + VLM)
# ============================================================
from src.models.grounded_vlm import GroundedVLM

grounded_vlm = GroundedVLM(
    vlm=vlm,
    detector=detector,
    use_spatial=True,
)

# Quick test
test_sample = eval_dataset[0]
result = grounded_vlm.generate(test_sample["image"], test_sample["question"])
print("Grounded prompt (excerpt):")
print(result["prompt"][:500])
print("\nGrounded response:")
print(result["response"][:500])

In [ ]:
# ============================================================
# 2.2 Initialize Chain-of-Thought Prompting
# ============================================================
from src.mitigation.cot_prompting import CoTPromptBuilder

cot_builder = CoTPromptBuilder(default_mode="grounded")

# Test CoT prompt
test_detections = detector.detect(test_sample["image"])
test_scene = detector.format_as_scene_summary(test_detections)

cot_prompt = cot_builder.build_grounded_cot_prompt(
    question=test_sample["question"],
    scene_summary=test_scene,
    detections=test_detections,
)

print("CoT Prompt:")
print(cot_prompt[:800])

# Generate with CoT
cot_response = vlm.generate(test_sample["image"], cot_prompt)
print("\nCoT Response:")
print(cot_response[:500])

In [ ]:
# ============================================================
# 2.3 Initialize Self-Verification Module
# ============================================================
from src.mitigation.self_verification import SelfVerifier

self_verifier = SelfVerifier(max_verification_rounds=1)

# Test self-verification
detected_categories = detector.get_detected_categories(test_detections)

verify_result = self_verifier.verify_and_correct(
    vlm=vlm,
    image=test_sample["image"],
    original_response=cot_response,
    detected_objects=detected_categories,
    scene_summary=test_scene,
)

print(f"Was corrected: {verify_result['was_corrected']}")
if verify_result["questions"]:
    print(f"Verification questions ({len(verify_result['questions'])}):\n")
    for q in verify_result["questions"]:
        print(f"  [{q['type']}] {q['concern']}")
print(f"\nCorrected response:\n{verify_result['corrected_response'][:500]}")

---
## Module 3: GRPO Training
Fine-tune the VLM with hallucination-aware rewards using Group Relative Policy Optimization.

In [ ]:
# ============================================================
# 3.1 Initialize Reward Function
# ============================================================
from src.training.reward_function import HallucinationReward

reward_fn = HallucinationReward(
    correct_object_weight=cfg["rewards"]["correct_object"],
    hallucinated_object_weight=cfg["rewards"]["hallucinated_object"],
    correct_attribute_weight=cfg["rewards"]["correct_attribute"],
    wrong_attribute_weight=cfg["rewards"]["wrong_attribute"],
    correct_spatial_weight=cfg["rewards"]["correct_spatial"],
    wrong_spatial_weight=cfg["rewards"]["wrong_spatial"],
    correct_count_weight=cfg["rewards"]["correct_count"],
    wrong_count_weight=cfg["rewards"]["wrong_count"],
    verbosity_penalty=cfg["rewards"]["verbosity_penalty"],
    safety_bonus=cfg["rewards"]["safety_bonus"],
)

# Test reward on baseline response
test_gt = test_sample["ground_truth"]
test_reward = reward_fn.compute_reward(
    response=cot_response,
    gt_objects=test_gt["unique_objects"],
    gt_counts=test_gt["object_counts"],
)

print("Reward Breakdown:")
for k, v in test_reward.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.3f}")
    elif isinstance(v, set):
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 3.2 Initialize GRPO Trainer
# ============================================================
from src.training.grpo_trainer import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    num_candidates=cfg["grpo"]["num_candidates"],
    learning_rate=cfg["grpo"]["learning_rate"],
    num_epochs=cfg["grpo"]["num_epochs"],
    batch_size=cfg["grpo"]["batch_size"],
    gradient_accumulation_steps=cfg["grpo"]["gradient_accumulation_steps"],
    max_grad_norm=cfg["grpo"]["max_grad_norm"],
    warmup_ratio=cfg["grpo"]["warmup_ratio"],
    kl_coeff=cfg["grpo"]["kl_coeff"],
    clip_range=cfg["grpo"]["clip_range"],
    max_new_tokens=cfg["grpo"].get("max_new_tokens", 256),
    save_steps=cfg["grpo"].get("save_steps", 50),
    eval_steps=cfg["grpo"].get("eval_steps", 25),
    lora_r=cfg["lora"]["r"],
    lora_alpha=cfg["lora"]["lora_alpha"],
    lora_dropout=cfg["lora"]["lora_dropout"],
    lora_target_modules=cfg["lora"]["target_modules"],
    output_dir="./results/grpo",
)

trainer = GRPOTrainer(
    model=vlm.get_model(),
    processor=vlm.get_processor(),
    reward_fn=reward_fn,
    config=grpo_config,
)

print(f"GRPO Trainer initialized.")
print(f"  Candidates per input: {grpo_config.num_candidates}")
print(f"  Epochs: {grpo_config.num_epochs}")
print(f"  Learning rate: {grpo_config.learning_rate}")
print(f"  Max new tokens: {grpo_config.max_new_tokens}")
print(f"  LoRA rank: {grpo_config.lora_r}")

In [ ]:
# ============================================================
# 3.3 Run GRPO Training
# ============================================================
# NOTE: This is the main training loop.
# Each step: K candidate generations + 2K log-prob passes.
# With K=2, max_new_tokens=256: ~80-100s/step on A100.

total_steps = len(train_dataset) * grpo_config.num_epochs
secs_per_step = 90  # conservative estimate for K=2, 256 tokens on A100
est_hours = total_steps * secs_per_step / 3600

print("Starting GRPO training...")
print(f"Training on {len(train_dataset)} samples, evaluating on {len(eval_dataset)} samples.")
print(f"Total steps: {total_steps} ({grpo_config.num_candidates} candidates/step, {grpo_config.max_new_tokens} max tokens)")
print(f"Estimated time: ~{est_hours:.1f} hours on A100\n")

training_logs = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    grounded_vlm=grounded_vlm,
)

print(f"\nTraining complete! {len(training_logs)} steps logged.")

---
## Module 4: Ablation Studies & Evaluation
Compare all pipeline configurations to understand the contribution of each component.

In [ ]:
# ============================================================
# 4.1 Run Full Ablation Study
# ============================================================
from src.evaluation.ablation import AblationRunner

ablation_runner = AblationRunner(
    vlm_baseline=vlm,
    grounded_vlm=grounded_vlm,
    cot_builder=cot_builder,
    self_verifier=self_verifier,
    detector=detector,
    metrics=metrics_calculator,
    output_dir="./results/ablation",
)

# Run all configurations
NUM_ABLATION_EVAL = 100  # Samples per configuration

ablation_results = ablation_runner.run_all_configs(
    dataset=eval_dataset,
    max_samples=NUM_ABLATION_EVAL,
)

In [ ]:
# ============================================================
# 4.2 POPE Benchmark Evaluation
# ============================================================
from src.data.pope_loader import load_pope_benchmarks, evaluate_pope_predictions

pope_benchmarks = load_pope_benchmarks(
    pope_dir=cfg["data"]["pope_dir"],
    images_dir=cfg["data"]["coco_images_dir"],
    max_samples_per_cat=200,  # Reduced for speed; use 500+ for final run
)

pope_results = {}
for cat, benchmark in pope_benchmarks.items():
    print(f"\nEvaluating POPE-{cat} ({len(benchmark)} questions)...")
    predictions = []
    labels = []

    for idx in tqdm(range(len(benchmark))):
        sample = benchmark[idx]
        # Use grounded VLM for POPE evaluation
        result = grounded_vlm.generate(sample["image"], sample["question"])
        predictions.append(result["response"])
        labels.append(sample["label"])

    pope_results[cat] = evaluate_pope_predictions(predictions, labels)
    print(f"  POPE-{cat}: {pope_results[cat]}")

print("\n=== POPE Results Summary ===")
for cat, res in pope_results.items():
    print(f"  {cat}: Acc={res['accuracy']:.3f}, F1={res['f1']:.3f}, Yes%={res['yes_ratio']:.3f}")

---
## Module 5: Results Visualization
Generate publication-quality figures for the dissertation.

In [ ]:
# ============================================================
# 5.1 Ablation Comparison Chart
# ============================================================
from src.evaluation.visualization import (
    plot_ablation_comparison,
    plot_training_curves,
    plot_hallucination_breakdown,
    create_results_table,
    plot_pope_results,
)

os.makedirs("./results", exist_ok=True)

# Ablation bar chart
plot_ablation_comparison(
    results=ablation_results,
    output_path="./results/ablation_comparison.png",
)

# Display inline
from IPython.display import Image as IPImage, display
display(IPImage(filename="./results/ablation_comparison.png"))

In [ ]:
# ============================================================
# 5.2 GRPO Training Curves
# ============================================================
plot_training_curves(
    training_logs=training_logs,
    output_path="./results/training_curves.png",
)

display(IPImage(filename="./results/training_curves.png"))

In [ ]:
# ============================================================
# 5.3 POPE Results Chart
# ============================================================
if pope_results:
    plot_pope_results(
        pope_results=pope_results,
        output_path="./results/pope_results.png",
    )
    display(IPImage(filename="./results/pope_results.png"))

In [ ]:
# ============================================================
# 5.4 Results Summary Table
# ============================================================
df = create_results_table(
    results=ablation_results,
    output_path="./results/results_table.csv",
)

# Display nicely
from IPython.display import HTML
display(HTML(df.to_html(index=False)))

In [ ]:
# ============================================================
# 5.5 Hallucination Breakdown (Baseline)
# ============================================================
import json

# Load baseline details if saved
baseline_detail_path = "./results/ablation/baseline_details.json"
if os.path.exists(baseline_detail_path):
    with open(baseline_detail_path) as f:
        baseline_details = json.load(f)
    plot_hallucination_breakdown(
        sample_details=baseline_details,
        output_path="./results/hallucination_breakdown.png",
    )
    display(IPImage(filename="./results/hallucination_breakdown.png"))
else:
    print("Run ablation study first to generate baseline details.")

In [ ]:
# ============================================================
# 5.6 Save All Results for Dissertation
# ============================================================
import json

# Compile all results
all_results = {
    "baseline_aggregate": baseline_agg,
    "ablation_results": ablation_results,
    "pope_results": pope_results,
    "grpo_training_steps": len(training_logs),
    "config": cfg,
}

with open("./results/all_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print("All results saved to ./results/")
print("\nFiles generated:")
for f in sorted(os.listdir("./results")):
    print(f"  {f}")

---
## Summary

This notebook executed the complete hallucination mitigation pipeline:

| Module | Status | Description |
|--------|--------|-------------|
| Module 1 | ✅ | Baseline evaluation — measured hallucination rates before mitigation |
| Module 2 | ✅ | Applied detection grounding, CoT prompting, self-verification |
| Module 3 | ✅ | GRPO training with hallucination-aware rewards |
| Module 4 | ✅ | Ablation studies + POPE benchmark evaluation |
| Module 5 | ✅ | Generated all publication-quality visualizations |

All results are saved in `./results/` for dissertation writing.